In [5]:
from pathlib import Path
import pandas as pd
import requests

from pymongo import MongoClient

**Repérage des données.** Le notebook peut être lancé depuis différents dossiers : cette recherche remonte donc jusqu'au dossier du projet contenant `data`, puis signale clairement si ce dossier est introuvable.

In [1]:
from pathlib import Path

# Chercher le dossier data depuis le notebook
racine = Path.cwd()

while racine != racine.parent and not (racine / "data").exists():
    racine = racine.parent

if not (racine / "data").exists():
    raise FileNotFoundError(
        "Le dossier data est introuvable."
    )

dossier_data = racine / "data"

print("Dossier du projet :", racine)
print("Dossier data :", dossier_data)

Dossier du projet : C:\Users\yanni\Desktop\SDV\2026-2027\Big_Data\Projet\cours_big_data
Dossier data : C:\Users\yanni\Desktop\SDV\2026-2027\Big_Data\Projet\cours_big_data\data


In [2]:
fichiers_csv = list(dossier_data.rglob("*.csv"))

for fichier in fichiers_csv:
    print(fichier)

C:\Users\yanni\Desktop\SDV\2026-2027\Big_Data\Projet\cours_big_data\data\dataset_nettoye\commandes_ecommerce.csv
C:\Users\yanni\Desktop\SDV\2026-2027\Big_Data\Projet\cours_big_data\data\processed\commandes_ecommerce.csv
C:\Users\yanni\Desktop\SDV\2026-2027\Big_Data\Projet\cours_big_data\data\processed\commandes_ecommerce_nettoyees.csv
C:\Users\yanni\Desktop\SDV\2026-2027\Big_Data\Projet\cours_big_data\data\processed\paniers.csv
C:\Users\yanni\Desktop\SDV\2026-2027\Big_Data\Projet\cours_big_data\data\processed\produits.csv
C:\Users\yanni\Desktop\SDV\2026-2027\Big_Data\Projet\cours_big_data\data\processed\utilisateurs_reduits.csv
C:\Users\yanni\Desktop\SDV\2026-2027\Big_Data\Projet\cours_big_data\data\raw\commandes_ecommerce.csv


In [3]:
for fichier in fichiers_csv:
    print(fichier.name)

commandes_ecommerce.csv
commandes_ecommerce.csv
commandes_ecommerce_nettoyees.csv
paniers.csv
produits.csv
utilisateurs_reduits.csv
commandes_ecommerce.csv


**Chargement d'un fichier de commandes.** Le premier CSV trouvé est lu dans un DataFrame afin de pouvoir examiner ensuite sa structure et ses données.

In [6]:
chemin_csv = fichiers_csv[0]

df_commandes = pd.read_csv(chemin_csv)

print("Fichier utilisé :", chemin_csv)
print("Dimensions :", df_commandes.shape)

Fichier utilisé : C:\Users\yanni\Desktop\SDV\2026-2027\Big_Data\Projet\cours_big_data\data\dataset_nettoye\commandes_ecommerce.csv
Dimensions : (800, 16)


In [7]:
print(df_commandes.columns.tolist())

['order_id', 'cart_id', 'user_id', 'username', 'product_id', 'product_title', 'category', 'brand', 'quantity', 'unit_price', 'line_total', 'discount_percentage', 'discounted_line_total', 'cart_total', 'cart_discounted_total', 'order_date']


In [8]:
df_commandes.head()

,order_id,cart_id,user_id,username,product_id,product_title,category,brand,quantity,unit_price,line_total,discount_percentage,discounted_line_total,cart_total,cart_discounted_total,order_date
0,CART-00001,1,1,emilys,162,Blue Frock,tops,NaN,4,29.99,119.96,12.13,105.41,13037.88,11510.81,2025-03-02
1,CART-00001,1,1,emilys,113,Generic Motorcycle,motorcycle,Generic Motors,3,3999.99,11999.97,12.10,10547.97,13037.88,11510.81,2025-06-13
2,CART-00001,1,1,emilys,122,iPhone 6,smartphones,Apple,3,299.99,899.97,6.69,839.76,13037.88,11510.81,2025-09-15
3,CART-00001,1,1,emilys,138,Baseball Ball,sports-accessories,NaN,2,8.99,17.98,1.71,17.67,13037.88,11510.81,2025-09-20
4,CART-00002,2,2,michaelw,86,Man Short Sleeve Shirt,mens-shirts,Casual Comfort,5,19.99,99.95,6.83,93.12,139.93,125.70,2025-11-28


**Connexion à MongoDB.** Cette étape vérifie que le serveur local répond, puis sélectionne la base `commerce` qui recevra les données.

In [9]:
client = MongoClient("mongodb://localhost:27017")

# Vérifier que MongoDB répond
print(client.admin.command("ping"))

# Sélectionner la base de données
db = client["commerce"]

print("Base de données :", db.name)

{'ok': 1.0}
Base de données : commerce


**Fonction de récupération de l'API.** Cette fonction télécharge une ressource DummyJSON, vérifie le statut HTTP et la présence de la clé attendue, puis renvoie sa liste de documents.

In [11]:
BASE_URL = "https://dummyjson.com"

def recuperer_ressource(nom):
    url = f"{BASE_URL}/{nom}"

    response = requests.get(
        url,
        params={"limit": 0},
        timeout=30
    )

    response.raise_for_status()

    donnees = response.json()

    if nom not in donnees:
        raise ValueError(
            f"Format inattendu pour la ressource {nom}"
        )

    return donnees[nom]

In [12]:
produits = recuperer_ressource("products")

print("Nombre de produits :", len(produits))

print(produits[0])

Nombre de produits : 194
{'id': 1, 'title': 'Essence Mascara Lash Princess', 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.', 'category': 'beauty', 'price': 9.99, 'discountPercentage': 10.48, 'rating': 2.56, 'stock': 99, 'tags': ['beauty', 'mascara'], 'brand': 'Essence', 'sku': 'BEA-ESS-ESS-001', 'weight': 4, 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99}, 'warrantyInformation': '1 week warranty', 'shippingInformation': 'Ships in 3-5 business days', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 3, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Eleanor Collins', 'reviewerEmail': 'eleanor.collins@x.dummyjson.com'}, {'rating': 4, 'comment': 'Very satisfied!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Lucas Gordon', 'reviewerEmail': 'lucas.gordon@x.dummyjson.com'}, {

**Correspondance API–MongoDB.** Ce dictionnaire associe chaque ressource téléchargée au nom de la collection MongoDB où elle sera stockée.

In [13]:
correspondance = {
    "products": "raw_products",
    "carts": "raw_carts",
    "users": "raw_users",
    "posts": "raw_posts",
    "comments": "raw_comments",
    "recipes": "raw_recipes",
    "quotes": "raw_quotes",
    "todos": "raw_todos"
}

**Ingestion des ressources API.** Pour chaque ressource, les documents sont récupérés, l'ancienne collection est vidée pour éviter les doublons lors d'une relance, puis les nouveaux documents sont insérés.

In [14]:
for ressource, collection_nom in correspondance.items():

    documents = recuperer_ressource(ressource)

    collection = db[collection_nom]

    # Nettoyer la collection avant une nouvelle ingestion
    collection.delete_many({})

    if documents:
        collection.insert_many(documents)

    print(
        f"{collection_nom} : "
        f"{collection.count_documents({})} documents"
    )

raw_products : 194 documents
raw_carts : 208 documents
raw_users : 208 documents
raw_posts : 251 documents
raw_comments : 340 documents
raw_recipes : 50 documents
raw_quotes : 1454 documents
raw_todos : 254 documents


In [15]:
for collection_nom in correspondance.values():

    collection = db[collection_nom]

    print(
        collection_nom,
        "->",
        collection.count_documents({})
    )

raw_products -> 194
raw_carts -> 208
raw_users -> 208
raw_posts -> 251
raw_comments -> 340
raw_recipes -> 50
raw_quotes -> 1454
raw_todos -> 254


In [16]:
db.raw_products.find_one()

{'_id': ObjectId('6b08093c218068d2be82edc0'),
 'id': 1,
 'title': 'Essence Mascara Lash Princess',
 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.',
 'category': 'beauty',
 'price': 9.99,
 'discountPercentage': 10.48,
 'rating': 2.56,
 'stock': 99,
 'tags': ['beauty', 'mascara'],
 'brand': 'Essence',
 'sku': 'BEA-ESS-ESS-001',
 'weight': 4,
 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99},
 'warrantyInformation': '1 week warranty',
 'shippingInformation': 'Ships in 3-5 business days',
 'availabilityStatus': 'In Stock',
 'reviews': [{'rating': 3,
   'comment': 'Would not recommend!',
   'date': '2025-04-30T09:41:02.053Z',
   'reviewerName': 'Eleanor Collins',
   'reviewerEmail': 'eleanor.collins@x.dummyjson.com'},
  {'rating': 4,
   'comment': 'Very satisfied!',
   'date': '2025-04-30T09:41:02.053Z',
   'reviewerName': 'Lucas 

In [17]:
db.raw_carts.find_one()

{'_id': ObjectId('6b08093c218068d2be82ee82'),
 'id': 1,
 'products': [{'id': 162,
   'title': 'Blue Frock',
   'price': 29.99,
   'quantity': 4,
   'total': 119.96,
   'discountPercentage': 12.13,
   'discountedTotal': 105.41,
   'thumbnail': 'https://cdn.dummyjson.com/product-images/tops/blue-frock/thumbnail.webp'},
  {'id': 113,
   'title': 'Generic Motorcycle',
   'price': 3999.99,
   'quantity': 3,
   'total': 11999.97,
   'discountPercentage': 12.1,
   'discountedTotal': 10547.97,
   'thumbnail': 'https://cdn.dummyjson.com/product-images/motorcycle/generic-motorcycle/thumbnail.webp'},
  {'id': 122,
   'title': 'iPhone 6',
   'price': 299.99,
   'quantity': 3,
   'total': 899.97,
   'discountPercentage': 6.69,
   'discountedTotal': 839.76,
   'thumbnail': 'https://cdn.dummyjson.com/product-images/smartphones/iphone-6/thumbnail.webp'},
  {'id': 138,
   'title': 'Baseball Ball',
   'price': 8.99,
   'quantity': 2,
   'total': 17.98,
   'discountPercentage': 1.71,
   'discountedTotal'

In [18]:
print("Nombre de lignes :", len(df_commandes))
print("Nombre de colonnes :", len(df_commandes.columns))

Nombre de lignes : 800
Nombre de colonnes : 16


**Import des commandes dans MongoDB.** Le DataFrame est converti en documents, l'ancienne collection est remplacée et les documents sont insérés dans `raw_commandes`.

In [19]:
collection_commandes = db["raw_commandes"]

# Nettoyer la collection avant un nouvel import
collection_commandes.delete_many({})

# Convertir le DataFrame en documents MongoDB
documents_commandes = df_commandes.to_dict(
    orient="records"
)

# Insérer les documents
if documents_commandes:
    resultat = collection_commandes.insert_many(
        documents_commandes
    )

print("Nombre de documents importés :", len(resultat.inserted_ids))

Nombre de documents importés : 800


In [20]:
print(
    "Nombre de documents MongoDB :",
    db.raw_commandes.count_documents({})
)

Nombre de documents MongoDB : 800


In [21]:
db.raw_commandes.find_one()

{'_id': ObjectId('6b080989218068d2be82f94f'),
 'order_id': 'CART-00001',
 'cart_id': 1,
 'user_id': 1,
 'username': 'emilys',
 'product_id': 162,
 'product_title': 'Blue Frock',
 'category': 'tops',
 'brand': nan,
 'quantity': 4,
 'unit_price': 29.99,
 'line_total': 119.96,
 'discount_percentage': 12.13,
 'discounted_line_total': 105.41,
 'cart_total': 13037.88,
 'cart_discounted_total': 11510.81,
 'order_date': '2025-03-02'}

In [22]:
print("Nombre de lignes :", len(df_commandes))

print(
    "Nombre de paniers distincts :",
    df_commandes["cart_id"].nunique()
)

Nombre de lignes : 800
Nombre de paniers distincts : 208


In [23]:
print(df_commandes.columns.tolist())

['order_id', 'cart_id', 'user_id', 'username', 'product_id', 'product_title', 'category', 'brand', 'quantity', 'unit_price', 'line_total', 'discount_percentage', 'discounted_line_total', 'cart_total', 'cart_discounted_total', 'order_date']


In [24]:
df_commandes.head(10)

,order_id,cart_id,user_id,username,product_id,product_title,category,brand,quantity,unit_price,line_total,discount_percentage,discounted_line_total,cart_total,cart_discounted_total,order_date
0,CART-00001,1,1,emilys,162,Blue Frock,tops,NaN,4,29.99,119.96,12.13,105.41,13037.88,11510.81,2025-03-02
1,CART-00001,1,1,emilys,113,Generic Motorcycle,motorcycle,Generic Motors,3,3999.99,11999.97,12.10,10547.97,13037.88,11510.81,2025-06-13
2,CART-00001,1,1,emilys,122,iPhone 6,smartphones,Apple,3,299.99,899.97,6.69,839.76,13037.88,11510.81,2025-09-15
3,CART-00001,1,1,emilys,138,Baseball Ball,sports-accessories,NaN,2,8.99,17.98,1.71,17.67,13037.88,11510.81,2025-09-20
4,CART-00002,2,2,michaelw,86,Man Short Sleeve Shirt,mens-shirts,Casual Comfort,5,19.99,99.95,6.83,93.12,139.93,125.70,2025-11-28
5,CART-00002,2,2,michaelw,104,Apple iPhone Charger,mobile-accessories,Apple,2,19.99,39.98,18.52,32.58,139.93,125.70,2025-02-22
6,CART-00003,3,3,sophiab,24,Fish Steak,groceries,NaN,1,14.99,14.99,4.23,14.36,1794.85,1625.77,2025-04-25
7,CART-00003,3,3,sophiab,123,iPhone 13 Pro,smartphones,Apple,1,1099.99,1099.99,9.37,996.92,1794.85,1625.77,2025-11-04
8,CART-00003,3,3,sophiab,129,Realme X,smartphones,Realme,1,299.99,299.99,6.95,279.14,1794.85,1625.77,2025-11-15
9,CART-00003,3,3,sophiab,86,Man Short Sleeve Shirt,mens-shirts,Casual Comfort,5,19.99,99.95,6.83,93.12,1794.85,1625.77,2025-10-12


In [25]:
df_commandes.describe(include="all")

,order_id,cart_id,user_id,username,product_id,product_title,category,brand,quantity,unit_price,line_total,discount_percentage,discounted_line_total,cart_total,cart_discounted_total,order_date
count,800,800.00000,800.00000,800,800.000000,800,800,423,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800
unique,208,NaN,NaN,208,NaN,188,24,63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,325
top,CART-00003,NaN,NaN,sophiab,NaN,Cat Food,kitchen-accessories,Apple,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-28
freq,6,NaN,NaN,6,NaN,12,119,65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
mean,NaN,105.36250,105.36250,NaN,94.863750,NaN,NaN,NaN,3.021250,1558.956125,4792.848288,10.597113,4320.886975,21491.209200,19337.097350,NaN
std,NaN,59.94717,59.94717,NaN,55.969978,NaN,NaN,NaN,1.412725,5414.429867,18490.875045,5.359502,16544.445879,38015.101234,33855.220883,NaN
min,NaN,1.00000,1.00000,NaN,1.000000,NaN,NaN,NaN,1.000000,0.790000,0.990000,0.040000,0.840000,6.970000,6.650000,NaN
25%,NaN,52.00000,52.00000,NaN,46.000000,NaN,NaN,NaN,2.000000,12.990000,29.980000,6.545000,26.147500,655.800000,603.927500,NaN
50%,NaN,105.50000,105.50000,NaN,96.000000,NaN,NaN,NaN,3.000000,34.990000,99.955000,11.050000,86.930000,2675.800000,2499.070000,NaN
75%,NaN,155.25000,155.25000,NaN,142.000000,NaN,NaN,NaN,4.000000,299.990000,519.960000,14.870000,451.457500,23114.790000,21783.540000,NaN


In [26]:
df_commandes.columns.tolist()

['order_id',
 'cart_id',
 'user_id',
 'username',
 'product_id',
 'product_title',
 'category',
 'brand',
 'quantity',
 'unit_price',
 'line_total',
 'discount_percentage',
 'discounted_line_total',
 'cart_total',
 'cart_discounted_total',
 'order_date']

In [27]:
df_commandes.head(10)

,order_id,cart_id,user_id,username,product_id,product_title,category,brand,quantity,unit_price,line_total,discount_percentage,discounted_line_total,cart_total,cart_discounted_total,order_date
0,CART-00001,1,1,emilys,162,Blue Frock,tops,NaN,4,29.99,119.96,12.13,105.41,13037.88,11510.81,2025-03-02
1,CART-00001,1,1,emilys,113,Generic Motorcycle,motorcycle,Generic Motors,3,3999.99,11999.97,12.10,10547.97,13037.88,11510.81,2025-06-13
2,CART-00001,1,1,emilys,122,iPhone 6,smartphones,Apple,3,299.99,899.97,6.69,839.76,13037.88,11510.81,2025-09-15
3,CART-00001,1,1,emilys,138,Baseball Ball,sports-accessories,NaN,2,8.99,17.98,1.71,17.67,13037.88,11510.81,2025-09-20
4,CART-00002,2,2,michaelw,86,Man Short Sleeve Shirt,mens-shirts,Casual Comfort,5,19.99,99.95,6.83,93.12,139.93,125.70,2025-11-28
5,CART-00002,2,2,michaelw,104,Apple iPhone Charger,mobile-accessories,Apple,2,19.99,39.98,18.52,32.58,139.93,125.70,2025-02-22
6,CART-00003,3,3,sophiab,24,Fish Steak,groceries,NaN,1,14.99,14.99,4.23,14.36,1794.85,1625.77,2025-04-25
7,CART-00003,3,3,sophiab,123,iPhone 13 Pro,smartphones,Apple,1,1099.99,1099.99,9.37,996.92,1794.85,1625.77,2025-11-04
8,CART-00003,3,3,sophiab,129,Realme X,smartphones,Realme,1,299.99,299.99,6.95,279.14,1794.85,1625.77,2025-11-15
9,CART-00003,3,3,sophiab,86,Man Short Sleeve Shirt,mens-shirts,Casual Comfort,5,19.99,99.95,6.83,93.12,1794.85,1625.77,2025-10-12


**Vérification des champs nécessaires.** On contrôle que le fichier contient les colonnes utilisées pour construire une table de paniers et simuler les paiements.

In [28]:
colonnes_paiement = [
    "cart_id",
    "user_id",
    "username",
    "order_id",
    "cart_discounted_total",
    "order_date"
]

for colonne in colonnes_paiement:
    print(
        colonne,
        "->",
        colonne in df_commandes.columns
    )

cart_id -> True
user_id -> True
username -> True
order_id -> True
cart_discounted_total -> True
order_date -> True


**Contrôle de cohérence par panier.** Le regroupement compte, pour chaque panier, le nombre de clients, de montants et de commandes distincts afin de repérer les valeurs qui devraient être uniques.

In [29]:
controle_paniers = (
    df_commandes
    .groupby("cart_id")
    .agg(
        nombre_clients=("user_id", "nunique"),
        nombre_montants=("cart_discounted_total", "nunique"),
        nombre_commandes=("order_id", "nunique")
    )
)

controle_paniers.head()

,nombre_clients,nombre_montants,nombre_commandes
cart_id,,,
1,1,1,1
2,1,1,1
3,1,1,1
4,1,1,1
5,1,1,1


**Repérage des paniers incohérents.** On conserve les paniers associés à plusieurs clients ou à plusieurs montants, puis on en compte les occurrences pour évaluer la qualité des données.

In [30]:
paniers_incoherents = controle_paniers[
    (controle_paniers["nombre_clients"] != 1)
    | (controle_paniers["nombre_montants"] != 1)
]

print(
    "Nombre de paniers incohérents :",
    len(paniers_incoherents)
)

Nombre de paniers incohérents : 0


**Création d'une ligne par panier.** Les colonnes utiles aux paiements sont sélectionnées et les doublons de `cart_id` sont retirés pour obtenir une table de paniers unique.

In [31]:
df_paniers = (
    df_commandes[
        [
            "cart_id",
            "user_id",
            "username",
            "order_id",
            "cart_discounted_total",
            "order_date"
        ]
    ]
    .drop_duplicates(subset=["cart_id"])
    .copy()
)

In [32]:
print("Nombre de paniers :", len(df_paniers))

print(
    "Nombre de paniers distincts :",
    df_paniers["cart_id"].nunique()
)

Nombre de paniers : 208
Nombre de paniers distincts : 208


In [33]:
df_paniers.head()

,cart_id,user_id,username,order_id,cart_discounted_total,order_date
0,1,1,emilys,CART-00001,11510.81,2025-03-02
4,2,2,michaelw,CART-00002,125.70,2025-11-28
6,3,3,sophiab,CART-00003,1625.77,2025-04-25
12,4,4,jamesd,CART-00004,636.81,2025-10-08
14,5,5,emmaj,CART-00005,1205.80,2025-10-28


**Simulation des statuts de paiement.** Chaque panier reçoit un statut aléatoire selon les probabilités définies. La graine fixe rend la simulation reproductible à chaque exécution.

In [34]:
import random

random.seed(42)

statuts = ["PAID", "REFUSED", "PENDING"]

probabilites = [0.70, 0.20, 0.10]

df_paniers["payment_status"] = [
    random.choices(
        statuts,
        weights=probabilites,
        k=1
    )[0]
    for _ in range(len(df_paniers))
]

**Simulation des statuts de paiement.** Cette cellule reprend la même simulation avec la même graine : elle produit donc les mêmes statuts que la cellule précédente.

In [35]:
import random

random.seed(42)

statuts = ["PAID", "REFUSED", "PENDING"]

probabilites = [0.70, 0.20, 0.10]

df_paniers["payment_status"] = [
    random.choices(
        statuts,
        weights=probabilites,
        k=1
    )[0]
    for _ in range(len(df_paniers))
]

**Préparation des montants de paiement.** Le montant total du panier est renommé `payment_amount` pour donner un nom adapté à la table des paiements; les autres colonnes sont conservées.

In [36]:
df_paiements = df_paniers.rename(
    columns={
        "cart_discounted_total": "payment_amount"
    }
).copy()

**Création d'identifiants de paiement.** Un identifiant séquentiel au format `PAY-0001` est attribué à chaque ligne afin de distinguer les paiements.

In [37]:
df_paiements["payment_id"] = [
    f"PAY-{i:04d}"
    for i in range(1, len(df_paiements) + 1)
]

In [38]:
print(df_paiements.columns.tolist())

['cart_id', 'user_id', 'username', 'order_id', 'payment_amount', 'order_date', 'payment_status', 'payment_id']


**Enregistrement des paiements dans MongoDB.** Les lignes du DataFrame sont converties en documents, puis insérées dans `raw_payments` après nettoyage de cette collection.

In [39]:
collection_paiements = db["raw_payments"]

# Nettoyer la collection avant une nouvelle simulation
collection_paiements.delete_many({})

# Convertir les lignes en documents MongoDB
documents_paiements = df_paiements.to_dict(
    orient="records"
)

# Insérer les documents
resultat = collection_paiements.insert_many(
    documents_paiements
)

print(
    "Nombre de paiements insérés :",
    len(resultat.inserted_ids)
)

Nombre de paiements insérés : 208


In [40]:
print(
    "Nombre total de paiements :",
    collection_paiements.count_documents({})
)

print(
    "Nombre de paniers distincts :",
    len(df_paiements["cart_id"].unique())
)

Nombre total de paiements : 208
Nombre de paniers distincts : 208


In [41]:
doublons = (
    df_paiements["cart_id"]
    .duplicated()
    .sum()
)

print("Nombre de paniers en doublon :", doublons)

Nombre de paniers en doublon : 0


**Synthèse par statut.** Le regroupement calcule, pour chaque statut de paiement, le nombre de paiements et le montant total associé.

In [42]:
analyse_paiements = (
    df_paiements
    .groupby("payment_status")
    .agg(
        nombre_paiements=("payment_id", "count"),
        montant_total=("payment_amount", "sum")
    )
)

analyse_paiements

,nombre_paiements,montant_total
payment_status,,
PAID,150,2669235.61
PENDING,21,363918.22
REFUSED,37,423555.75


**Calcul du chiffre d'affaires encaissé.** Seuls les montants dont le statut est `PAID` sont additionnés; le résultat est arrondi à deux décimales.

In [43]:
chiffre_affaires_encaisse = df_paiements.loc[
    df_paiements["payment_status"] == "PAID",
    "payment_amount"
].sum()

print(
    "Chiffre d'affaires encaissé :",
    round(chiffre_affaires_encaisse, 2)
)

Chiffre d'affaires encaissé : 2669235.61


**Calcul du taux de refus.** Le nombre de paiements `REFUSED` est rapporté au nombre total de paniers, puis exprimé en pourcentage.

In [44]:
taux_refus = (
    df_paiements["payment_status"].eq("REFUSED").sum()
    / len(df_paiements)
)

print("Taux de refus :", round(taux_refus * 100, 2), "%")

Taux de refus : 17.79 %
